In [0]:
#dbutils.library.restartPython()

In [0]:
from common_utils.bronze import read_raw, bronze_ingestor

from common_utils.logging import get_logger


logger = get_logger("ds2b")


sources = {
    "s3_sales": {
        "raw_folder":"s3_sales",
        "file_format":"parquet",
        "read_options":{},
        "target_table":"retaildataplatform.bronze.s3_sales",
        "mode":"overwrite"
    },

    "cosmosdb_sales_orders": {
        "raw_folder":"cosmosdb_sales_orders",
        "file_format":"json",
        "read_options":{},
        "target_table":"retaildataplatform.bronze.cosmosdb_sales_orders",
        "mode":"overwrite"
    },
    "sqlserver_customers": {
        "raw_folder":"sqlserver_customers",
        "file_format":"csv",
        "read_options":{"header":"true", "inferSchema":"true"},
        "target_table":"retaildataplatform.bronze.sqlserver_customers",
        "mode":"overwrite"
    }

}

run_date = "2026-09-21"

for source_name, config in sources.items():
    raw_path = f"/Volumes/retaildataplatform/bronze/raw_data/{config['raw_folder']}/load_date={run_date}"
    logger.info("[%s] reading from path %s", source_name, raw_path)
    df = read_raw(spark, raw_path, file_format=config['file_format'], options = config['read_options'])
    
    rows = bronze_ingestor(df, mode=config['mode'], target_table=config['target_table'])
    logger.info("[%s] ingested %s rows", source_name, rows)


logger.info("bronze finished for all sources")

In [0]:
raw_path = "/Volumes/retaildataplatform/bronze/raw_data/cosmosdb_sales_orders/load_date=2026-09-07/"
target_table = "retaildataplatform.bronze.cosmosdb_sales_orders"


df_cosmos = read_raw(spark, raw_path, file_format="json", options = None)
bronze_ingestor(df_cosmos, mode="overwrite", target_table=target_table)

logger.info("Bronze Completed for %s ", target_table)


In [0]:
raw_path = "/Volumes/retaildataplatform/bronze/raw_data/sales/"
target_table = "retaildataplatform.bronze.s3_sales"



df_s3 = read_raw(spark, raw_path, file_format="parquet", options = None)
bronze_ingestor(df_s3, mode="overwrite", target_table=target_table)

logger.info("Bronze Completed for %s ", target_table)


In [0]:
raw_path = "/Volumes/retaildataplatform/bronze/raw_data/sqlserver_customers/load_date=2026-09-07/"
target_table = "retaildataplatform.bronze.sqlserver_customers"
options = {"header":"true", "inferSchema":"true"}


df_sqlserver = read_raw(spark, raw_path, file_format="csv", options = options)
bronze_ingestor(df_sqlserver, mode="overwrite", target_table=target_table)

logger.info("Bronze Completed for %s ", target_table)
